# SONARA Training V6: corrected CNN-BiLSTM, binary baselines, external test

**Run `SONARA_DATASET_V2.ipynb` first.** This notebook replaces `SONARA_NEW_V5.ipynb`. Use a GPU runtime (Runtime > Change runtime type > T4 GPU).

| Fix | V5 behaviour | V6 behaviour |
| --- | --- | --- |
| BiLSTM pooling | Used the LSTM output at the last time step, where the backward direction has seen one frame | Concatenates the final forward and final backward hidden states |
| Best weights in multi-seed runs | `best_state = model.state_dict()` kept a reference, so the "best" model was the last epoch | `copy.deepcopy(...)`, and weights are also saved to Drive |
| Threshold | Tuned to maximize validation **accuracy**, which lowered distress recall | Highest threshold whose validation **distress recall** is at least `TARGET_RECALL`; results at 0.5 are also reported |
| Data | PVSD pooled into train/val/test | Train, validate and test on SmartEars only; PVSD is an external test, scored per file |
| Split | Random split per seed, model and augmentation randomness not seeded per run | Grouped, stratified split per seed; every run seeds Python, NumPy and PyTorch |
| Baselines | None trained in the notebook; DS-CNN was 3-class | CNN front end only (ablation), binary DS-CNN, MobileNetV3-Small, majority class; all on the same splits and seeds |
| Metrics | Accuracy, macro-F1, distress F1, ROC-AUC | Adds distress precision/recall, FNR, FPR, PR-AUC, confusion matrix, errors by original subclass, per-file PVSD results |
| Augmentation | Masks filled with the mean of all three channels; gain multiplied dB values | Masks filled with each channel's own mean; gain is a dB offset on the log-mel channel |
| Runtime | Not measured | Parameter breakdown, model size, CPU latency proxy, and exported weights for `pi_latency_benchmark.py` |
| Audio loading | `torchaudio.load` (fails without `torchcodec` on recent Colab) | Handled in Dataset V2 with `soundfile` |

**Runtime estimate:** 4 models × 5 seeds. Expect several hours on a T4; the CNN-BiLSTM is the slowest. Results are saved after every run, so if Colab disconnects, rerun the notebook and finished runs are skipped. You can also run one model at a time by editing `MODELS_TO_RUN`.

**Hyperparameters are unchanged from V5** (Adam, lr 1e-4, weight decay 1e-4, batch 16, dropout 0.3, label smoothing 0.02, at most 50 epochs, early stopping patience 8), and the same values are used for every model. They were set by hand, not tuned.

In [ ]:
import os, sys, json, copy, time, random, hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, average_precision_score, confusion_matrix)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# ---------------- CONFIG ----------------
DATA_DIR    = os.environ.get("SONARA_OUT", "/content/drive/MyDrive/SONARA/v2")   # output of Dataset V2
RESULTS_DIR = os.path.join(DATA_DIR, "results_v6")

MODELS_TO_RUN = ["cnn_bilstm", "cnn_frontend_only", "dscnn", "mobilenetv3_small"]
# Add "cnn_bilstm_laststep" to measure the effect of the pooling fix (V5 behaviour).
SEEDS = [7, 21, 42, 77, 99]

EPOCHS, PATIENCE = 50, 8
BATCH_SIZE = 16
LR, WEIGHT_DECAY = 1e-4, 1e-4
LABEL_SMOOTHING = 0.02
TARGET_RECALL = 0.90          # decision threshold = highest one with validation distress recall >= this
WINDOW_FRAMES = 400           # 4-s model input
NUM_WORKERS = 2

SMOKE_TEST = os.environ.get("SONARA_SMOKE") == "1"   # tiny settings for a quick code check only
if SMOKE_TEST:
    EPOCHS, PATIENCE, SEEDS, BATCH_SIZE, NUM_WORKERS = 2, 2, [7, 21], 8, 0
# ----------------------------------------

os.makedirs(RESULTS_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| torch", torch.__version__)
if device.type == "cpu" and not SMOKE_TEST:
    print("WARNING: no GPU. Switch the Colab runtime to a GPU before training.")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

## 1. Load the Dataset V2 features

In [ ]:
se_npz = np.load(os.path.join(DATA_DIR, "smartears_feats.npz"))
X_se, y_se = se_npz["X"], se_npz["y_bin"]
meta_se = pd.read_csv(os.path.join(DATA_DIR, "smartears_meta.csv"))
pv_npz = np.load(os.path.join(DATA_DIR, "pvsd_feats.npz"))
X_pv, y_pv_win = pv_npz["X"], pv_npz["y_bin"]
meta_pv_win = pd.read_csv(os.path.join(DATA_DIR, "pvsd_windows.csv"))
ds_cfg = json.load(open(os.path.join(DATA_DIR, "dataset_config.json")))

assert len(X_se) == len(meta_se) and (meta_se.y_bin.values == y_se).all()
assert len(X_pv) == len(meta_pv_win)
print("SmartEars:", X_se.shape, "| Non-distress / Distress:", np.bincount(y_se))
print(meta_se.groupby("orig_class").size().to_dict())
print("PVSD windows:", X_pv.shape, "| files:", meta_pv_win.file_idx.nunique())

## 2. Grouped, stratified split (about 71 / 14 / 14)

`StratifiedGroupKFold` keeps every group (one file, or one recording if `GROUP_PATTERN` was set in Dataset V2) in a single split and keeps the class ratio similar across splits. One fold of 7 becomes the test set; one fold of 6 from the remainder becomes validation. The assertions below fail loudly if any group appears in two splits.

In [ ]:
def make_split(seed, y=y_se, groups=meta_se.group.values):
    idx = np.arange(len(y))
    outer = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=seed)
    rest, test = next(outer.split(idx, y, groups))
    inner = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=seed)
    tr, va = next(inner.split(rest, y[rest], groups[rest]))
    train, val = rest[tr], rest[va]
    g = lambda a: set(groups[a])
    assert not (g(train) & g(val)) and not (g(train) & g(test)) and not (g(val) & g(test)), "group leakage"
    return train, val, test

split_fingerprints = {}
for s in SEEDS:
    tr, va, te = make_split(s)
    split_fingerprints[s] = hashlib.md5(np.sort(te).tobytes()).hexdigest()[:8]
    print(f"seed {s}: train {len(tr)} {np.bincount(y_se[tr])} | val {len(va)} {np.bincount(y_se[va])} "
          f"| test {len(te)} {np.bincount(y_se[te])} | test-set id {split_fingerprints[s]}")
assert len(set(split_fingerprints.values())) == len(SEEDS), "two seeds produced the same test set"

## 3. Dataset with corrected augmentation

* Training: random 4-s crop from the stored 5 s, then augmentation.
* Validation, test and PVSD: centre 4 s (PVSD windows are already 4 s), no augmentation.
* All randomness uses PyTorch's generator, so it is reproducible per seed, including inside DataLoader workers.

In [ ]:
class SonaraDataset(Dataset):
    def __init__(self, X, y, indices=None, train=False):
        self.X, self.y = X, y
        self.idx = np.arange(len(y)) if indices is None else np.asarray(indices)
        self.train = train

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        x = torch.from_numpy(self.X[j].astype(np.float32))      # (3, 64, T)
        t = x.shape[-1]
        if t > WINDOW_FRAMES:
            s = int(torch.randint(0, t - WINDOW_FRAMES + 1, (1,))) if self.train else (t - WINDOW_FRAMES) // 2
            x = x[:, :, s:s + WINDOW_FRAMES]
        if self.train:
            x = self.augment(x)
        return x, torch.tensor(int(self.y[j]))

    @staticmethod
    def augment(x):
        x = x.clone()
        c, f, t = x.shape
        ch_mean = x.mean(dim=(1, 2), keepdim=True)              # each channel's own mean
        if torch.rand(1) < 0.4:                                  # time mask
            w = int(torch.randint(5, max(6, t // 8 + 1), (1,)))
            t0 = int(torch.randint(0, t - w + 1, (1,)))
            x[:, :, t0:t0 + w] = ch_mean.expand(c, f, w)
        if torch.rand(1) < 0.4:                                  # frequency mask
            h = int(torch.randint(3, max(4, f // 8 + 1), (1,)))
            f0 = int(torch.randint(0, f - h + 1, (1,)))
            x[:, f0:f0 + h, :] = ch_mean.expand(c, h, t)
        if torch.rand(1) < 0.2:                                  # loudness change: +/- 6 dB
            x[0] = x[0] + (torch.rand(1) * 12 - 6)               # a constant offset leaves deltas unchanged
        return x

def make_loader(X, y, idx, train, seed):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(SonaraDataset(X, y, idx, train), batch_size=BATCH_SIZE, shuffle=train,
                      num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"), generator=g,
                      worker_init_fn=lambda w: np.random.seed((seed * 1000 + w) % 2**32))

## 4. Models

Same code as `sonara_models.py`, which the Pi benchmark script imports. The parameter breakdown below answers the panel's question about where the 1,221,186 parameters come from (matrix row 15).

In [ ]:
"""SONARA model definitions (shared by SONARA_TRAIN_V6.ipynb and pi_latency_benchmark.py).

All models take a feature tensor of shape (batch, 3, 64, frames):
channel 0 = log-mel spectrogram (dB), channel 1 = delta, channel 2 = delta-delta.
All models output 2 logits: index 0 = Non-distress, index 1 = Distress.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class CNN_BiLSTM(nn.Module):
    """CNN front end + bidirectional LSTM.

    pooling="final_states" (default, FIXED): concatenates the final hidden state of the
        forward direction (has read the whole sequence left to right) and of the backward
        direction (has read the whole sequence right to left).
    pooling="last_step" (LEGACY, the V5 behaviour): takes the LSTM output at the last time
        step. The backward half of that output has only seen one frame, so the model
        barely uses bidirectional context. Kept only to measure the effect of the fix.

    Attribute names (conv1, bn1, conv2, bn2, lstm, fc) match the V5 notebook, so a V5
    state_dict loads into pooling="last_step".
    """

    def __init__(self, n_mels=64, num_classes=2, cnn_channels=64, lstm_hidden=128,
                 lstm_layers=1, dropout=0.3, pooling="final_states"):
        super().__init__()
        if pooling not in ("final_states", "last_step"):
            raise ValueError("pooling must be 'final_states' or 'last_step'")
        self.pooling = pooling
        self.conv1 = nn.Conv2d(3, cnn_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(cnn_channels)
        self.conv2 = nn.Conv2d(cnn_channels, cnn_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(cnn_channels)
        self.pool = nn.MaxPool2d(kernel_size=(2, 2))
        freq_out = n_mels // 4                      # two 2x2 poolings: 64 -> 16
        # Built here (not lazily inside forward) so the model can be loaded directly on the Pi.
        self.lstm = nn.LSTM(
            input_size=cnn_channels * freq_out,     # 64 * 16 = 1024 inputs per time step
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_hidden * 2, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        b, c, fq, t = x.shape                        # (B, 64, 16, T/4)
        x = x.permute(0, 3, 1, 2).reshape(b, t, c * fq)   # sequence of T/4 steps
        out, (h_n, _) = self.lstm(x)
        if self.pooling == "final_states":
            z = torch.cat([h_n[-2], h_n[-1]], dim=1)  # final forward + final backward state
        else:
            z = out[:, -1, :]                         # legacy V5 behaviour
        return self.fc(self.dropout(z))


class CNNFrontEndOnly(nn.Module):
    """Ablation: exactly the CNN-BiLSTM front end, with global average pooling instead of the
    BiLSTM. Comparing this with CNN_BiLSTM shows what the BiLSTM adds."""

    def __init__(self, n_mels=64, num_classes=2, cnn_channels=64, dropout=0.3):
        super().__init__()
        self.conv1 = nn.Conv2d(3, cnn_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(cnn_channels)
        self.conv2 = nn.Conv2d(cnn_channels, cnn_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(cnn_channels)
        self.pool = nn.MaxPool2d(kernel_size=(2, 2))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(cnn_channels, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        z = self.gap(x).flatten(1)
        return self.fc(self.dropout(z))


class _DSBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.dw = nn.Conv2d(ch, ch, kernel_size=3, padding=1, groups=ch, bias=False)
        self.bn_dw = nn.BatchNorm2d(ch)
        self.pw = nn.Conv2d(ch, ch, kernel_size=1, bias=False)
        self.bn_pw = nn.BatchNorm2d(ch)

    def forward(self, x):
        x = F.relu(self.bn_dw(self.dw(x)))
        return F.relu(self.bn_pw(self.pw(x)))


class DSCNN(nn.Module):
    """Depthwise-separable CNN in the style of Zhang et al. (2017), "Hello Edge: Keyword
    Spotting on Microcontrollers". This is a REIMPLEMENTATION. If the team still has the
    original DS-CNN code from Table 3.1, use that instead and keep the same training loop."""

    def __init__(self, num_classes=2, channels=172, n_blocks=4, dropout=0.3):
        super().__init__()
        self.conv = nn.Conv2d(3, channels, kernel_size=(10, 4), stride=(2, 2), padding=(5, 1), bias=False)
        self.bn = nn.BatchNorm2d(channels)
        self.blocks = nn.Sequential(*[_DSBlock(channels) for _ in range(n_blocks)])
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(channels, num_classes)

    def forward(self, x):
        x = F.relu(self.bn(self.conv(x)))
        x = self.blocks(x)
        z = self.gap(x).flatten(1)
        return self.fc(self.dropout(z))


def mobilenet_v3_small(num_classes=2):
    """MobileNetV3-Small from torchvision, trained from scratch (no ImageNet weights).
    Qiao et al. (2026) evaluated MobileNetV3 on SmartEars, so this gives a published
    architecture run on OUR split."""
    from torchvision.models import mobilenet_v3_small as _mnv3
    return _mnv3(weights=None, num_classes=num_classes)


MODEL_BUILDERS = {
    "cnn_bilstm": lambda: CNN_BiLSTM(pooling="final_states"),
    "cnn_bilstm_laststep": lambda: CNN_BiLSTM(pooling="last_step"),   # legacy, for comparison only
    "cnn_frontend_only": lambda: CNNFrontEndOnly(),
    "dscnn": lambda: DSCNN(),
    "mobilenetv3_small": lambda: mobilenet_v3_small(),
}

MODEL_LABELS = {
    "cnn_bilstm": "CNN-BiLSTM (fixed pooling)",
    "cnn_bilstm_laststep": "CNN-BiLSTM (V5 last-step pooling)",
    "cnn_frontend_only": "CNN front end only (no LSTM)",
    "dscnn": "DS-CNN (binary)",
    "mobilenetv3_small": "MobileNetV3-Small",
}


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def param_breakdown(model):
    """Parameters per top-level layer, for the panel's 'how were the parameters computed' question."""
    rows = []
    total = count_params(model)
    for name, module in model.named_children():
        n = sum(p.numel() for p in module.parameters())
        if n:
            rows.append((name, n, 100.0 * n / total))
    return rows, total


for name in MODELS_TO_RUN + ["cnn_bilstm_laststep"]:
    m = MODEL_BUILDERS[name]()
    print(f"{MODEL_LABELS[name]:36s} {count_params(m):>10,} parameters  "
          f"({count_params(m) * 4 / 1e6:.2f} MB as float32)")

rows, total = param_breakdown(CNN_BiLSTM())
print("\nCNN-BiLSTM parameter breakdown:")
for name, n, pct in rows:
    print(f"  {name:8s} {n:>10,}  {pct:5.1f}%")
print(f"  {'total':8s} {total:>10,}")

## 5. Training and evaluation functions

In [ ]:
def train_one(model_name, seed, train_idx, val_idx):
    set_seed(seed)
    model = MODEL_BUILDERS[model_name]().to(device)
    counts = np.bincount(y_se[train_idx], minlength=2)
    class_weights = torch.tensor(counts.mean() / counts, dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2, min_lr=1e-6)
    tl = make_loader(X_se, y_se, train_idx, True, seed)
    vl = make_loader(X_se, y_se, val_idx, False, seed)

    best_loss, best_state, best_epoch, bad, history = float("inf"), None, 0, 0, []
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        tr_loss = tr_n = 0
        for xb, yb in tl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
            tr_loss += loss.item() * len(yb); tr_n += len(yb)
        model.eval()
        va_loss = va_n = va_correct = 0
        with torch.no_grad():
            for xb, yb in vl:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                va_loss += criterion(out, yb).item() * len(yb); va_n += len(yb)
                va_correct += (out.argmax(1) == yb).sum().item()
        tr_loss, va_loss = tr_loss / tr_n, va_loss / va_n
        sched.step(va_loss)
        improved = va_loss < best_loss - 1e-4
        if improved:
            best_loss, best_epoch, bad = va_loss, epoch, 0
            best_state = copy.deepcopy(model.state_dict())     # a real copy, not a reference
        else:
            bad += 1
        history.append(dict(epoch=epoch, train_loss=tr_loss, val_loss=va_loss, val_acc=va_correct / va_n,
                            lr=opt.param_groups[0]["lr"]))
        print(f"  [{model_name} seed {seed}] epoch {epoch:02d} train {tr_loss:.4f} val {va_loss:.4f} "
              f"acc {va_correct / va_n:.4f}{' *' if improved else ''}")
        if bad >= PATIENCE:
            break
    model.load_state_dict(best_state)
    model.eval()
    info = dict(best_epoch=best_epoch, epochs_run=epoch, best_val_loss=best_loss,
                train_seconds=time.time() - t0, class_weights=class_weights.tolist(),
                train_counts=counts.tolist())
    return model, info, pd.DataFrame(history)


@torch.no_grad()
def predict(model, X, y, idx=None):
    loader = DataLoader(SonaraDataset(X, y, idx, train=False), batch_size=64, shuffle=False, num_workers=0)
    probs = []
    for xb, _ in loader:
        probs.append(F.softmax(model(xb.to(device)), dim=1)[:, 1].cpu())
    return torch.cat(probs).numpy()


def choose_threshold(y_true, p, target_recall=TARGET_RECALL):
    # Highest threshold whose distress recall is still >= target (best precision at that recall).
    cands = [t for t in np.round(np.arange(0.01, 0.995, 0.005), 3)
             if recall_score(y_true, (p >= t).astype(int), zero_division=0) >= target_recall]
    return float(max(cands)) if cands else 0.5


def metrics(y_true, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    two = len(np.unique(y_true)) == 2
    return dict(threshold=thr, n=int(len(y_true)), accuracy=accuracy_score(y_true, pred),
                macro_f1=f1_score(y_true, pred, average="macro", zero_division=0),
                distress_precision=precision_score(y_true, pred, zero_division=0),
                distress_recall=recall_score(y_true, pred, zero_division=0),
                distress_f1=f1_score(y_true, pred, zero_division=0),
                fnr=fn / (fn + tp) if fn + tp else float("nan"),
                fpr=fp / (fp + tn) if fp + tn else float("nan"),
                roc_auc=roc_auc_score(y_true, p) if two else float("nan"),
                pr_auc=average_precision_score(y_true, p) if two else float("nan"),
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))

## 6. Run every model on every seed

Each finished run writes `<model>_seed<seed>.json`, test predictions with file paths, per-file PVSD predictions, the training history and the best weights to `RESULTS_DIR`. Finished runs are skipped when you rerun the cell.

In [ ]:
def run(model_name, seed):
    tag = f"{model_name}_seed{seed}"
    out_json = os.path.join(RESULTS_DIR, tag + ".json")
    if os.path.exists(out_json) and not SMOKE_TEST:
        print("skip (done):", tag)
        return json.load(open(out_json))
    train_idx, val_idx, test_idx = make_split(seed)
    model, info, hist = train_one(model_name, seed, train_idx, val_idx)

    p_val = predict(model, X_se, y_se, val_idx)
    thr = choose_threshold(y_se[val_idx], p_val)
    p_test = predict(model, X_se, y_se, test_idx)
    y_test = y_se[test_idx]

    # External test: PVSD, never seen in training. Window level and file level (mean probability).
    p_pv = predict(model, X_pv, y_pv_win)
    pv = meta_pv_win.assign(prob_distress=p_pv)
    pv_file = pv.groupby(["file_idx", "filename", "orig_class"], as_index=False).agg(
        y_bin=("y_bin", "first"), prob_distress=("prob_distress", "mean"), n_windows=("prob_distress", "size"))

    # Errors by original subclass: share of clips flagged as distress.
    te = meta_se.iloc[test_idx][["filepath", "orig_class", "y_bin"]].assign(prob_distress=p_test)
    te["pred_bin"] = (te.prob_distress >= thr).astype(int)
    sub = te.groupby("orig_class").agg(n=("pred_bin", "size"), flagged_distress=("pred_bin", "mean")).round(4)

    res = dict(model=model_name, label=MODEL_LABELS[model_name], seed=seed, params=count_params(model),
               test_set_id=split_fingerprints.get(seed), split_sizes=[len(train_idx), len(val_idx), len(test_idx)],
               **info, val=metrics(y_se[val_idx], p_val, thr),
               test=metrics(y_test, p_test, thr), test_at_0_5=metrics(y_test, p_test, 0.5),
               pvsd_file=metrics(pv_file.y_bin.values, pv_file.prob_distress.values, thr),
               pvsd_window=metrics(pv.y_bin.values, p_pv, thr),
               test_by_subclass=sub.to_dict(orient="index"))
    te.to_csv(os.path.join(RESULTS_DIR, tag + "_test_predictions.csv"), index=False)
    pv_file.to_csv(os.path.join(RESULTS_DIR, tag + "_pvsd_files.csv"), index=False)
    hist.to_csv(os.path.join(RESULTS_DIR, tag + "_history.csv"), index=False)
    torch.save(model.state_dict(), os.path.join(RESULTS_DIR, tag + ".pt"))
    json.dump(res, open(out_json, "w"), indent=2, default=float)
    t = res["test"]
    print(f"DONE {tag}: thr {thr:.3f} | test acc {t['accuracy']:.4f} recall {t['distress_recall']:.4f} "
          f"F1 {t['distress_f1']:.4f} AUC {t['roc_auc']:.4f} | PVSD file AUC {res['pvsd_file']['roc_auc']:.4f}")
    return res

all_results = []
for model_name in MODELS_TO_RUN:
    for seed in SEEDS:
        all_results.append(run(model_name, seed))

## 7. Summary table (for the new Table 3.1)

Mean ± standard deviation over seeds. **Test** = SmartEars held-out test set at the recall-targeted threshold. **PVSD** = external test, per file. The majority-class row is the floor any model must beat.

In [ ]:
def pm(series, digits=3):
    return f"{series.mean():.{digits}f} ± {series.std(ddof=1):.{digits}f}" if len(series) > 1 else f"{series.mean():.{digits}f}"

done = [json.load(open(os.path.join(RESULTS_DIR, f))) for f in sorted(os.listdir(RESULTS_DIR)) if f.endswith(".json")]
flat = pd.json_normalize(done, sep=".")
flat = flat[flat.seed.isin(SEEDS)]

cols = {"test.accuracy": "Accuracy", "test.macro_f1": "Macro-F1", "test.distress_recall": "Distress recall",
        "test.distress_precision": "Distress precision", "test.distress_f1": "Distress F1",
        "test.fnr": "FNR", "test.roc_auc": "ROC-AUC", "pvsd_file.roc_auc": "PVSD ROC-AUC (file)",
        "pvsd_file.distress_recall": "PVSD recall (file)", "pvsd_file.fpr": "PVSD FPR (file)"}
summary = []
for m, g in flat.groupby("model", sort=False):
    row = {"Model": g.label.iloc[0], "Parameters": f"{int(g.params.iloc[0]):,}", "Seeds": len(g),
           "Epochs (best)": pm(g.best_epoch, 1), "Threshold": pm(g["test.threshold"])}
    row.update({v: pm(g[k]) for k, v in cols.items()})
    summary.append(row)

# Majority class (always Non-distress), on the same test sets.
maj = [metrics(y_se[make_split(s)[2]], np.zeros(len(make_split(s)[2])), 0.5) for s in SEEDS]
maj = pd.DataFrame(maj)
summary.insert(0, {"Model": "Majority class (always Non-distress)", "Parameters": "0", "Seeds": len(SEEDS),
                   "Accuracy": pm(maj.accuracy), "Macro-F1": pm(maj.macro_f1), "Distress recall": "0.000",
                   "Distress F1": "0.000", "FNR": "1.000"})
summary = pd.DataFrame(summary)
summary = summary[["Model", "Parameters", "Seeds", "Epochs (best)", "Threshold"] + list(cols.values())]
summary.to_csv(os.path.join(RESULTS_DIR, "summary_table.csv"), index=False)
print(summary.to_string(index=False))

In [ ]:
# Paired comparison: same seed = same split, so compare model pairs seed by seed.
def paired(a, b, metric="test.distress_f1"):
    A = flat[flat.model == a].set_index("seed")[metric]
    B = flat[flat.model == b].set_index("seed")[metric]
    common = A.index.intersection(B.index)
    if len(common) == 0:
        return None
    d = (A[common] - B[common])
    return dict(pair=f"{a} - {b}", metric=metric, seeds=len(common), mean_diff=round(d.mean(), 4),
                sd_diff=round(d.std(ddof=1), 4) if len(d) > 1 else None, wins=int((d > 0).sum()))

pairs = []
for b in ["cnn_frontend_only", "dscnn", "mobilenetv3_small", "cnn_bilstm_laststep"]:
    for metric in ["test.distress_recall", "test.distress_f1", "test.roc_auc", "pvsd_file.roc_auc"]:
        r = paired("cnn_bilstm", b, metric)
        if r: pairs.append(r)
pairs = pd.DataFrame(pairs)
print(pairs.to_string(index=False))
print("\nRead 'wins' as: on how many of the seeds the CNN-BiLSTM scored higher. With 5 seeds, "
      "a 3-2 split is not evidence of a difference; report it as no clear difference.")
pairs.to_csv(os.path.join(RESULTS_DIR, "paired_comparison.csv"), index=False)

In [ ]:
# Errors by original subclass on the test set (matrix row 4): which Non-distress clips cause false alarms?
sub_rows = []
for r in done:
    if r["seed"] in SEEDS:
        for cls, v in r["test_by_subclass"].items():
            sub_rows.append(dict(model=r["label"], seed=r["seed"], subclass=cls, flagged_distress=v["flagged_distress"]))
sub_df = pd.DataFrame(sub_rows)
print(sub_df.groupby(["model", "subclass"]).flagged_distress.agg(["mean", "std"]).round(3))
print("\nFor 'sick' this is recall; for 'healthy' and 'none' it is the false-alarm rate of that subclass.")

## 8. Model size and CPU latency (proxy only)

This measures one 4-s clip on **this machine's CPU**. It is not the Raspberry Pi number. For Table 3.1, copy `sonara_models.py` and `pi_latency_benchmark.py` to the Pi and run `python3 pi_latency_benchmark.py`.

In [ ]:
torch.set_num_threads(max(1, os.cpu_count() or 1))
lat = []
for name in MODELS_TO_RUN:
    m = MODEL_BUILDERS[name]().eval()
    x = torch.randn(1, 3, 64, WINDOW_FRAMES)
    with torch.no_grad():
        for _ in range(5): m(x)
        ts = []
        for _ in range(3 if SMOKE_TEST else 50):
            t = time.perf_counter(); m(x); ts.append((time.perf_counter() - t) * 1000)
    lat.append(dict(model=MODEL_LABELS[name], params=count_params(m), size_mb=round(count_params(m) * 4 / 1e6, 2),
                    cpu_median_ms=round(float(np.median(ts)), 1), cpu_p95_ms=round(float(np.percentile(ts, 95)), 1)))
lat = pd.DataFrame(lat)
lat.to_csv(os.path.join(RESULTS_DIR, "latency_proxy_colab_cpu.csv"), index=False)
print(f"CPU threads: {torch.get_num_threads()} (Colab proxy, NOT the Pi)")
print(lat.to_string(index=False))

## 9. (Optional) Label smoothing check for matrix rows 7 and 8

Compares ε = 0, 0.02, 0.05 and 0.1 for the CNN-BiLSTM on **validation** data only (the test set is not used to choose settings). Takes about 12 extra training runs. Set `RUN_LS_ABLATION = True` to run it.

In [ ]:
RUN_LS_ABLATION = False
if RUN_LS_ABLATION:
    rows = []
    for eps in [0.0, 0.02, 0.05, 0.1]:
        LABEL_SMOOTHING = eps
        for seed in SEEDS[:3]:
            tr, va, _ = make_split(seed)
            model, info, _ = train_one("cnn_bilstm", seed, tr, va)
            p = predict(model, X_se, y_se, va)
            m = metrics(y_se[va], p, choose_threshold(y_se[va], p))
            rows.append(dict(epsilon=eps, seed=seed, val_loss=info["best_val_loss"], val_auc=m["roc_auc"],
                             val_distress_f1=m["distress_f1"], val_recall=m["distress_recall"]))
    LABEL_SMOOTHING = 0.02
    ls = pd.DataFrame(rows)
    ls.to_csv(os.path.join(RESULTS_DIR, "label_smoothing_ablation.csv"), index=False)
    print(ls.groupby("epsilon")[["val_auc", "val_distress_f1", "val_recall"]].agg(["mean", "std"]).round(4))
    print("Note: validation loss is not comparable across epsilon values, because smoothing changes the loss itself.")